In [ ]:
https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/viewer/en/train?row=35&views%5B%5D=en

In [ ]:

# %pip install sentencepiece pytorch_lightning peft transformers

In [1]:
# %pip install -U bitsandbytes

In [1]:
import os

import torch
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader

from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LlamaForCausalLM, LlamaTokenizer, default_data_collator, get_linear_schedule_with_warmup
# from bitsandbytes import BitsAndBytesConfig

from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

from datasets import load_dataset
from dataclasses import dataclass, field

from typing import List, Optional, Tuple



os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Specify the GPU you want to use



In [2]:
@dataclass
class Args:
    BATCH_SIZE: int = 1
    NUM_WORKERS: int = 15
    RANDOM_SEED: int = 42
    projectName: str = "LlamaFinetune"
    datasetName: str = "FreedomIntelligence/medical-o1-reasoning-SFT"
    modelName: str = 'Llama3.2-3B-Instruct-hf'

    # Model Parameters
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_steps: int = 100
    max_steps: int = 1000
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"
    ])

    
args = Args()

In [3]:
torch.set_float32_matmul_precision('high')
# Setting the seed
pl.seed_everything(args.RANDOM_SEED)
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
# print("Device:", device)

#  Path to the folder where the pretrained models are saved
HOME_PATH = os.path.dirname(os.getcwd())
CHECKPOINT_PATH = os.path.join(HOME_PATH, "saved_models", args.projectName)

device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

Seed set to 42


Device: cuda:1


In [4]:
os.path.dirname(os.getcwd())

'/home/home/Desktop/research'

In [5]:
model_path = os.path.join(HOME_PATH,"Models", args.modelName)
# Verify the path exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"The directory {model_path} does not exist. Please check the path.")

args.model_path = model_path

In [6]:
# # Login using e.g. `huggingface-cli login` to access this dataset
# ds = load_dataset(args.datasetName, "en")
# train_data = ds['train']

In [7]:
# i =  enumerate(train_data)
# print(next(i))

In [8]:
# train_data[0]

In [9]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

print(len(tokenizer))

# Set pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Add special tokens if not already added
special_tokens_dict = {"additional_special_tokens": ["<|user|>", "<|assistant|>"]}
num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)

print(f"Added {num_added_tokens} special tokens.")

tokenizer_size = len(tokenizer)

128256
Added 2 special tokens.


In [10]:
class ClassDataset(Dataset):
    def __init__(self, datasetName:str, lang:str = 'en', tokenizer = None, max_length: int =512, split: str = 'train'):
        ds = load_dataset(datasetName, lang)
        all_data = ds['train']
        
        # Use Huggingface's built-in split
        dataset_split = all_data.train_test_split(test_size=0.2, seed=42)
        train_data = dataset_split['train']
        val_data = dataset_split['test']

        if split == 'train':
            self.data = train_data
        elif split == 'validation':
            self.data = val_data
        else:
            raise ValueError(f"Unknown split: {split}")


        # self.data = ds['train']
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.create_tokens_text(self.data[index]) 
    
    def create_tokens_text(self, data_text):
        conversation = (
            f"<|user|>\n{data_text['Question']}\n"
            f"<|assistant|>\n{data_text['Complex_CoT']}\n\n"
            f"My response is:\n{data_text['Response']}"
        )


        # Tokenize
        encodings = self.tokenizer(conversation, 
                        truncation=True,
                        max_length=self.max_length,
                        padding="max_length",
                        return_tensors="pt"
                    )
    
        # Create labels (for causal language modeling)
        input_ids = encodings["input_ids"][0]
        attention_mask = encodings["attention_mask"][0]
        labels = input_ids.clone()

        # Mask labels for user prompts (optional)
        # This means we only calculate loss on assistant responses
        # Find positions of <|assistant|> tokens
        assistant_positions = []
        assistant_token_id =self.tokenizer.convert_tokens_to_ids("<|assistant|>")
        for i, token_id in enumerate(input_ids):
            if token_id == assistant_token_id:
                assistant_positions.append(i)

        # Set labels for non-assistant text to -100 (ignored in loss calculation)
        if assistant_positions:
            is_assistant = False
            for i in range(len(labels)):
                if i in assistant_positions:
                    is_assistant = True
                if not is_assistant:
                    labels[i] = -100

        return {
            # "text":conversation,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
            }

In [11]:
datasetTrain = ClassDataset(datasetName = args.datasetName, tokenizer=tokenizer, split="train", max_length=1024)
datasetVal = ClassDataset(datasetName = args.datasetName, tokenizer=tokenizer, split="validation", max_length=1024)

dataloader_train_ = DataLoader(
    dataset=datasetTrain,
    batch_size= args.BATCH_SIZE,
    num_workers= args.NUM_WORKERS,
    shuffle = True,
    pin_memory= True
)
    
dataloader_val_ = DataLoader(
    dataset=datasetVal,
    batch_size=args.BATCH_SIZE,
    num_workers=args.NUM_WORKERS,
    shuffle=False,
    pin_memory=True
)

In [12]:
class LoRAFineTuner(pl.LightningModule):
    def __init__(self, args,**kwargs):
        super().__init__()
        self.save_hyperparameters(ignore=['dataloader_train_', 'dataloader_val_', "tokenizer"])

        learning_rate = kwargs.get("learning_rate", 5e-5)
        weight_decay = kwargs.get("weight_decay", 0.01)
        warmup_steps = kwargs.get("warmup_steps", 100)
        max_steps = kwargs.get("max_steps", 100)
        num_added_tokens = kwargs.get("num_added_tokens", None)
        tokenizer_size = kwargs.get("num_tokenizer_sizeadded_tokens", None)

      

        # Load the model with the new quantization configuration
        self.model = LlamaForCausalLM.from_pretrained(
            pretrained_model_name_or_path=args.model_path,
            quantization_config={"load_in_8bit": True},  # Pass the quantization config here
            torch_dtype=torch.float16,
            # device_map="auto",  # You can uncomment this if you want automatic device allocation
        )    

        if self.hparams.num_added_tokens is not None:
        # # Resize model embeddings!
            if self.hparams.num_added_tokens > 0:
                self.model.resize_token_embeddings(self.hparams.tokenizer_size)   

      
        # Prepare model for training
        self.model = prepare_model_for_kbit_training(self.model)
        
        # Configure LoRA
        peft_config = LoraConfig(
            task_type = TaskType.CAUSAL_LM,
            inference_mode = False,
            r = args.lora_r,
            lora_alpha = args.lora_alpha,
            lora_dropout = args.lora_dropout,
            target_modules = args.lora_target_modules,
        )
        
        # Get PEFT model
        self.model = get_peft_model(self.model, peft_config)

        
        
        
        self.model.print_trainable_parameters()
    
    
    
    def configure_optimizers(self):
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": self.hparams.weight_decay,
            },
            {
                "params": [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
            },
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=self.hparams.learning_rate)
        
        scheduler = get_linear_schedule_with_warmup(
            optimizer, 
            num_warmup_steps=self.hparams.warmup_steps, 
            num_training_steps=self.hparams.max_steps
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
            },
        }
    
    def training_step(self, batch, batch_idx):
        raise NotImplementedError

    def validation_step(self, batch, batch_idx):
        raise NotImplementedError

    def test_step(self, batch, batch_idx):
        raise NotImplementedError
    
 
class fineTuneModel(LoRAFineTuner):

    def _calculate_loss_(self, batch,  mode="train")-> Tuple[torch.Tensor, float]:
        outputs = self.model(**batch)
        loss = outputs.loss
        self.log(f"{mode}_loss", loss, prog_bar=True)
        return loss
    
    def forward(self, **inputs):
        return self.model(**inputs)
    
    def training_step(self, batch, batch_idx):
        loss = self._calculate_loss_(batch=batch, mode="train")

        # Log learning rate
        current_lr = self.trainer.optimizers[0].param_groups[0]["lr"]
        self.log("lr", current_lr, on_step=True, on_epoch=False)

        return loss
    
    def validation_step(self, batch, batch_idx):
        loss = self._calculate_loss_(batch=batch, mode="val")
        return loss

In [13]:
# def train_finetuneModel(args, **kwargs):

#     name_test = kwargs.get("name_test", "nameTestProject")
#     monitor = kwargs.get("monitor", "val_loss")
#     max_epochs = kwargs.get("max_epochs", 10)
#     CHECKPOINT_PATH = kwargs.get("CHECKPOINT_PATH", 10)
#     dataloader_train_ = kwargs.get("dataloader_train_", None)
#     dataloader_val_ = kwargs.get("dataloader_val_", None)
#     tokenizer = kwargs.get("tokenizer", None)

#     # Create a PyTorch Lightning trainer with the generation callback
#     root_dir = os.path.join(CHECKPOINT_PATH, name_test)
#     os.makedirs(root_dir, exist_ok=True)


#     # Configure callbacks
#     checkpoint_callback = ModelCheckpoint(
#         # dirpath="checkpoints",
#         filename="llama-3.2-1b-lora-{epoch:02d}-{val_loss:.2f}",
#         # filename="llama-3.2-1b-lora-{epoch:02d}-{val_loss:.2f}",
#         save_top_k=2,
#         monitor= monitor,
#         mode="min",
#     )

#      # Configure logger
#     # logger = TensorBoardLogger("logs", name="llama-3.2-1b-lora")

#     # Configure trainer
#     trainer = pl.Trainer(
#         default_root_dir=root_dir,
#         max_epochs=max_epochs,
#         gradient_clip_val=1.0,
#         accumulate_grad_batches=64,  # Effective batch size = 4 * 4 = 16
#         precision="16-mixed",  # Use mixed precision for efficiency
#         # logger=logger,
#         callbacks=[checkpoint_callback],
#         log_every_n_steps=10,

#         accelerator="gpu",
#         devices=1,            # Automatically detect how many GPUs
#         strategy="auto",            # Let Lightning choose DDP or DataParallel
#         profiler="simple",
#         # fast_dev_run=True
#     )

#     # Train

#     model = fineTuneModel(args, **kwargs)
#     trainer.fit(model = model, train_dataloaders=dataloader_train_, val_dataloaders=dataloader_val_ )


#     # Save model and tokenizer after training
#     model_save_path = os.path.join(CHECKPOINT_PATH, name_test, "final_model")
#     os.makedirs(model_save_path, exist_ok=True)
    
#     # Save model
#     model.model.save_pretrained(model_save_path)
#     print(f"Model saved to {model_save_path}")
    
#     # Save tokenizer
#     model.model.config.save_pretrained(model_save_path)
#     tokenizer.save_pretrained(model_save_path)
#     print(f"Tokenizer saved to {model_save_path}")

In [14]:
# max_epochs = 20
# max_steps = len(dataloader_train_) * max_epochs
# warmup_steps = 0.25 * max_steps

# training_params = {
#     "name_test":"LlamaFinetune",
#     "monitor":"train_loss",
#     "max_epochs": 20,

#     "learning_rate" :3.5e-4,
#     "weight_decay": 0.01,
#     "warmup_steps": warmup_steps,
#     "max_steps": max_steps,

#     "num_added_tokens": num_added_tokens,
#     "tokenizer_size": tokenizer_size,
#     "CHECKPOINT_PATH":CHECKPOINT_PATH,
#     "dataloader_train_":dataloader_train_,
#     "dataloader_val_":dataloader_val_,
#     "tokenizer":tokenizer,

#     # "mode":"train" #finetune, test
# }

# train_finetuneModel(args=args, **training_params)

In [15]:
def train_finetuneModel(args, **kwargs):

    name_test = kwargs.get("name_test", "nameTestProject")
    monitor = kwargs.get("monitor", "val_loss")
    max_epochs = kwargs.get("max_epochs", 10)
    CHECKPOINT_PATH = kwargs.get("CHECKPOINT_PATH", 10)
    dataloader_train_ = kwargs.get("dataloader_train_", None)
    dataloader_val_ = kwargs.get("dataloader_val_", None)
    tokenizer = kwargs.get("tokenizer", None)
    mode = kwargs.get("mode", "train")

    # Create a PyTorch Lightning trainer with the generation callback
    root_dir = os.path.join(CHECKPOINT_PATH, name_test)
    os.makedirs(root_dir, exist_ok=True)


    # Configure callbacks
    checkpoint_callback = ModelCheckpoint(
        # dirpath="checkpoints",
        filename="llama-3.2-1b-lora-{epoch:02d}-{val_loss:.2f}",
        # filename="llama-3.2-1b-lora-{epoch:02d}-{val_loss:.2f}",
        save_top_k=2,
        monitor= monitor,
        mode="min",
    )

     # Configure logger
    # logger = TensorBoardLogger("logs", name="llama-3.2-1b-lora")

    # Configure trainer
    trainer = pl.Trainer(
        default_root_dir=root_dir,
        max_epochs=max_epochs,
        gradient_clip_val=1.0,
        accumulate_grad_batches=64,  # Effective batch size = 4 * 4 = 16
        precision="16-mixed",  # Use mixed precision for efficiency
        # logger=logger,
        callbacks=[checkpoint_callback],
        log_every_n_steps=10,

        accelerator="gpu",
        devices=1,            # Automatically detect how many GPUs
        strategy="auto",            # Let Lightning choose DDP or DataParallel
        profiler="simple",
        # fast_dev_run=True
    )

    if mode == "train":
    # Train
        model = fineTuneModel(args, **kwargs)
        trainer.fit(model = model, train_dataloaders=dataloader_train_, val_dataloaders=dataloader_val_ )
    
    elif mode =="finetune":
        pretrained_filename = os.path.join(
            CHECKPOINT_PATH,
            name_test,
            "lightning_logs",
            "version_3",
            "checkpoints",
            "llama-3.2-1b-lora-epoch=02-val_loss=0.94.ckpt"
        )
        
        if not os.path.exists(pretrained_filename):
            raise FileNotFoundError(f"The directory {pretrained_filename} does not exist. Please check the path.")
        
        model = fineTuneModel.load_from_checkpoint(pretrained_filename, args=args, **kwargs)
        trainer.fit(model= model, train_dataloaders=dataloader_train_, val_dataloaders=dataloader_val_)

    elif mode=="test":
        pretrained_filename = os.path.join(
            CHECKPOINT_PATH,
            name_test,
            "lightning_logs",
            "version_3",
            "checkpoints",
            "llama-3.2-1b-lora-epoch=02-val_loss=0.94.ckpt"
        )
        
        if not os.path.exists(pretrained_filename):
            raise FileNotFoundError(f"The directory {pretrained_filename} does not exist. Please check the path.")
        
        model = fineTuneModel.load_from_checkpoint(pretrained_filename, args=args, **kwargs)
        print("Model Load")
        

    # Save model and tokenizer after training
    model_save_path = os.path.join(CHECKPOINT_PATH, name_test, "final_model")
    os.makedirs(model_save_path, exist_ok=True)
    
    # Save model
    model.model.save_pretrained(model_save_path)
    print(f"Model saved to {model_save_path}")
    
    # Save tokenizer
    model.model.config.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    print(f"Tokenizer saved to {model_save_path}")

In [16]:
max_epochs = 20
max_steps = len(dataloader_train_) * max_epochs
warmup_steps = 0.25 * max_steps

training_params = {
    "name_test":"LlamaFinetune",
    "monitor":"train_loss",
    "max_epochs": 20,

    "learning_rate" :3.5e-4,
    "weight_decay": 0.01,
    "warmup_steps": warmup_steps,
    "max_steps": max_steps,

    "num_added_tokens": num_added_tokens,
    "tokenizer_size": tokenizer_size,
    "CHECKPOINT_PATH":CHECKPOINT_PATH,
    "dataloader_train_":dataloader_train_,
    "dataloader_val_":dataloader_val_,
    "tokenizer":tokenizer,

    "mode":"test" #finetune, train
}

train_finetuneModel(args=args, **training_params)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


trainable params: 12,156,928 || all params: 3,224,912,896 || trainable%: 0.3770
Model Load


/home/home/anaconda3/lib/python3.12/site-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Model saved to /home/home/Desktop/research/saved_models/LlamaFinetune/LlamaFinetune/final_model
Tokenizer saved to /home/home/Desktop/research/saved_models/LlamaFinetune/LlamaFinetune/final_model
